In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import AnalysisFunctions

AF = AnalysisFunctions.Analysis_Functions()

import colour

from colour_demosaicing import (
    ROOT_RESOURCES_EXAMPLES,
    demosaicing_CFA_Bayer_bilinear,
    demosaicing_CFA_Bayer_Malvar2004,
    demosaicing_CFA_Bayer_Menon2007,
    mosaicing_CFA_Bayer,
)

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

In [2]:
data_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [3]:
data_saving_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241029/data"
savename = "Cy3Cy5_Efittesting_E"

In [4]:
image_size = 26

from Camera_QE import getpixelefficiency

gpe = getpixelefficiency.GPE()
R, G, B, wavelength = gpe.getpixelefficiency("Camera_QE/CS505CU_QE.csv")
RGB = np.vstack([R, G, B]).T
filters = np.ones_like(wavelength)
masks = MSF.MF.get_masks(MSF.mosaic_unit, image_size, image_size)
masks_3d = np.dstack([masks["maskR"], masks["maskG"], masks["maskB"]])
wavelength = wavelength
absolute_QYs = np.vstack([B, G, R])
camera_calibration = {}
camera_calibration["gain"] = gain[:image_size, :image_size]
camera_calibration["offset"] = offset[:image_size, :image_size]
camera_calibration["variance"] = variance[:image_size, :image_size]
camera_calibration["readnoise"] = readnoise[:image_size, :image_size]
camera_calibration["rqe"] = rqe[:image_size, :image_size]

In [5]:
n_photon_lspace = np.linspace(500, 20000, 100)
E_space = np.linspace(0, 1, 21)
n_bootstrap = 1000
background_photons = 0
pixel_size = 69
NA = 1.49

In [6]:
fret_pair = ["Cy3", "Cy5"]
fret_QYs = np.array([0.15, 0.27])
camera_string = "CS505CU"
filter_string = "nofilters"
fret_spectral_model = AF.gen_E_lookuptable(
    fret_pair, fret_QYs, RGB, wavelength, filters, camera_string, filter_string
)

In [7]:
fret_pair_0 = [x for x in AF.dye_filenames if fret_pair[0] in x]
fret_pair_1 = [x for x in AF.dye_filenames if fret_pair[1] in x]
if "ATTO" not in fret_pair[0]:
    separator = ","
    skip_rows = 0
else:
    separator = "\t"
    skip_rows = 1
data_F0 = pl.read_csv(
    os.path.join(AF.dye_folder, fret_pair_0[0]),
    separator=separator,
    skip_rows=skip_rows,
)
F0_wl = data_F0[:, 0].to_numpy()
F0_em = np.multiply(
    filters,
    np.interp(x=wavelength, xp=F0_wl, fp=data_F0[:, 1].to_numpy(), left=0, right=0),
)
F0_em = F0_em / np.trapz(x=wavelength, y=F0_em)
if "ATTO" not in fret_pair[1]:
    separator = ","
    skip_rows = 0
else:
    separator = "\t"
    skip_rows = 1
data_F1 = pl.read_csv(
    os.path.join(AF.dye_folder, fret_pair_1[0]),
    separator=separator,
    skip_rows=skip_rows,
)
F1_wl = data_F1[:, 0].to_numpy()
F1_em = np.multiply(
    filters,
    np.interp(x=wavelength, xp=F1_wl, fp=data_F1[:, 1].to_numpy(), left=0, right=0),
)
F1_em = F1_em / np.trapz(x=wavelength, y=F1_em)

In [8]:
parameters = np.array(["xc", "yc", "A", "sigma", "b", "E"])

Espace = np.linspace(0, 1, 101)
for E in E_space:
    dyes = np.zeros([1, len(wavelength)])

    fit_RMSE_mean = np.zeros([len(parameters), len(n_photon_lspace)])
    fit_bias_mean = np.zeros([len(parameters), len(n_photon_lspace)])
    fit_RMSE_std = np.zeros([len(parameters), len(n_photon_lspace)])
    fit_bias_std = np.zeros([len(parameters), len(n_photon_lspace)])
    start = time.time()

    spectrum = ((F1_em * fret_QYs[1]) * E) + ((F0_em * fret_QYs[0]) * (1 - E))
    dyes[0, :] = spectrum / sum(spectrum)

    average_emission_wavelength = np.trapz(
        y=wavelength * (dyes.T / np.trapz(x=wavelength, y=dyes)).T, x=wavelength
    )
    sigma_PSF = PSF.diffraction_limit(average_emission_wavelength, NA)[0]

    x0y0 = {}
    max = pixel_size * image_size
    x0y0["dye"] = np.array([[max / 2], [max / 2]])

    expected_parameters = np.array([max / 2, max / 2, sigma_PSF, 0, E])
    real_params = pl.DataFrame(
        data=np.expand_dims(expected_parameters, 0),
        schema=list(parameters[[0, 1, 3, 4, 5]]),
    )
    real_params.write_csv(
        os.path.join(
            data_saving_folder,
            savename + "_" + str(E).replace(".", "p") + "_input_parameters.csv",
        )
    )

    for i, n_photon in enumerate(n_photon_lspace):
        expected_parameters = np.array([max / 2, max / 2, n_photon, sigma_PSF, 0, E])
        fit_values = np.zeros([len(parameters), n_bootstrap])
        n_photons = {}
        n_photons["dye"] = n_photon
        x0 = np.full(n_bootstrap, max / 2) + np.random.normal(size=n_bootstrap) * (
            pixel_size / 2
        )
        y0 = np.full(n_bootstrap, max / 2) + np.random.normal(size=n_bootstrap) * (
            pixel_size / 2
        )

        for j in np.arange(n_bootstrap):
            x0y0["dye"] = np.array([[x0[j]], [y0[j]]])
            ground_truth, bayer_image = MSF.gen_camera_images(
                camera_calibration,
                wavelength,
                absolute_QYs,
                dyes,
                n_photons,
                x0y0,
                background_photons=background_photons,
                NA=NA,
                pixel_size=pixel_size,
            )
            photoelectron_data = np.divide(
                np.divide(
                    np.subtract(bayer_image, camera_calibration["offset"]),
                    camera_calibration["gain"],
                ),
                camera_calibration["rqe"],
            )

            erd = sCMOS.var_weighted_uniform_filter(
                photoelectron_data, camera_calibration["variance"], 4
            )
            erd[erd < 0] = 0
            erd = erd + 1
            error_map = np.add(erd, np.square(camera_calibration["readnoise"]))
            weights_map = np.power(error_map, -2)

            xc_ig, yc_ig = np.unravel_index(
                np.argmax(
                    sCMOS.var_weighted_uniform_filter(
                        photoelectron_data, camera_calibration["variance"], 4
                    )
                ),
                photoelectron_data.shape,
            )
            A = np.sum((photoelectron_data))
            sigma = 3
            b = 0
            initial_guess = np.array([xc_ig, yc_ig, A, sigma, b, 0.5])
            result = AF.FRET_fit(
                photoelectron_data,
                initial_guess,
                masks=masks_3d,
                weights=weights_map,
                fret_spectral_model=fret_spectral_model,
                Espace=Espace,
                display=False,
            )
            if result.status < 0:
                fit_values[:, j] = np.full_like(result.x, np.NAN)
            else:
                fit_values[:, j] = result.x

        print(
            "Analysed photon flux {}/{}    Time elapsed: {:.3f} min".format(
                i + 1, len(n_photon_lspace), (time.time() - start) / 60.0
            ),
            end="\r",
            flush=True,
        )

        fit_values[[0, 1, 3], :] = fit_values[[0, 1, 3], :] * pixel_size

        for param in np.arange(len(parameters)):
            if param == 0:
                fit_RMSE_mean[param, i] = np.nanmean(
                    np.sqrt(np.square(fit_values[param, :] - y0))
                )
                fit_bias_mean[param, i] = np.nanmean(fit_values[param, :] - y0)
                fit_RMSE_std[param, i] = np.nanstd(
                    np.sqrt(np.square(fit_values[param, :] - y0))
                )
                fit_bias_std[param, i] = np.nanstd(fit_values[param, :] - y0)
            elif param == 1:
                fit_RMSE_mean[param, i] = np.nanmean(
                    np.sqrt(np.square(fit_values[param, :] - x0))
                )
                fit_bias_mean[param, i] = np.nanmean(fit_values[param, :] - x0)
                fit_RMSE_std[param, i] = np.nanstd(
                    np.sqrt(np.square(fit_values[param, :] - x0))
                )
                fit_bias_std[param, i] = np.nanstd(fit_values[param, :] - x0)
            else:
                fit_RMSE_mean[param, i] = np.nanmean(
                    np.sqrt(
                        np.square((fit_values[param, :]) - expected_parameters[param])
                    )
                )
                fit_RMSE_std[param, i] = np.nanstd(
                    np.sqrt(
                        np.square((fit_values[param, :]) - expected_parameters[param])
                    )
                )
                fit_bias_mean[param, i] = np.nanmean(
                    fit_values[param, :] - expected_parameters[param]
                )
                fit_bias_std[param, i] = np.nanstd(
                    fit_values[param, :] - expected_parameters[param]
                )

    parameters_tosave = np.array(["n_photons", "xc", "yc", "A", "sigma", "b", "E"])
    means = pl.DataFrame(
        data=np.vstack([n_photon_lspace, fit_RMSE_mean]), schema=list(parameters_tosave)
    )
    stds = pl.DataFrame(
        data=np.vstack([n_photon_lspace, fit_RMSE_std]), schema=list(parameters_tosave)
    )
    means_b = pl.DataFrame(
        data=np.vstack([n_photon_lspace, fit_bias_mean]), schema=list(parameters_tosave)
    )
    stds_b = pl.DataFrame(
        data=np.vstack([n_photon_lspace, fit_bias_std]), schema=list(parameters_tosave)
    )

    means.write_csv(
        os.path.join(
            data_saving_folder,
            savename
            + "_"
            + str(E).replace(".", "p")
            + "_69nmpixel_RMSE_mean_VUxyestimation.csv",
        )
    )
    stds.write_csv(
        os.path.join(
            data_saving_folder,
            savename
            + "_"
            + str(E).replace(".", "p")
            + "_69nmpixel_RMSE_std_VUxyestimation.csv",
        )
    )
    means.write_csv(
        os.path.join(
            data_saving_folder,
            savename
            + "_"
            + str(E).replace(".", "p")
            + "_69nmpixel_bias_mean_VUxyestimation.csv",
        )
    )
    stds.write_csv(
        os.path.join(
            data_saving_folder,
            savename
            + "_"
            + str(E).replace(".", "p")
            + "_69nmpixel_bias_std_VUxyestimation.csv",
        )
    )